## Load segments from Slicer

In [1]:
import numpy
import sys
import os
import matplotlib.pyplot as plt
# import skimage
import nrrd
from scipy import ndimage
from skimage import measure
import pyvista as pv

sys.path.insert(0, "../src")  # add Folder_2 path to search list

from meshing_functions import getSurfaceMesh, tetra_mesh_from_stl, tetra_shell_from_two_surfaces


pixel_spacing = numpy.loadtxt("/Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/VTI_0.5_fat_AIMax/pixel_spacing.txt")

base = "/Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/cropped T1/Meshes/"

out_folder = "alternative_re-mesh/"



In [2]:
eye_id = 3
others_id = [1]

mask = numpy.load(base +"segmentations_pdt1_v160725._filtered_surf_id=" +str(eye_id) + ".npy")
mask[mask==1] = eye_id

for id in others_id:
    mask_other = numpy.load(base +"segmentations_pdt1_v160725._filtered_surf_id=" +str(id) + ".npy")
    mask[mask_other==1] = id

verts, faces, _, _ = measure.marching_cubes(mask >=1)



### Option 1 - 3D mesh directly 

In [3]:
# grid = pv.ImageData()
# grid.dimensions = numpy.array(mask.shape) + 1
# grid.spacing = (1.0, 1.0, 1.0)
# grid.origin = (0.0, 0.0, 0.0)

# grid.cell_data["values"] = mask.flatten(order="F")

# # Extract domains
# obj1 = grid.threshold([0.5, 1.5], scalars="values")
# obj2 = grid.threshold([1.5, 2.5], scalars="values")

# # Combine domains
# combined = obj1 + obj2

# # Tetrahedral mesh
# tetra = combined.delaunay_3d(alpha=1.2)

# # Save
# tetra.save(base + out_folder + "mesh.vtk")

### Option 2 - Surface first

In [4]:
# grid = pv.ImageData()
# grid.dimensions = numpy.array(mask.shape) + 1
# grid.spacing = (1.0, 1.0, 1.0)
# grid.origin = (0.0, 0.0, 0.0)

# grid.cell_data["values"] = mask.flatten(order="F")

# # Step 1: Extract surfaces for each domain
# surf_obj1 = grid.threshold([3-0.1, 3+0.1], scalars="values").extract_surface()
# surf_obj2 = grid.threshold([1-0.1, 1+0.1], scalars="values").extract_surface()

# # Extract surfaces for each domain
# surf_obj1 = obj1.extract_surface()
# surf_obj2 = obj2.extract_surface()

# # Clean duplicates
# surf_obj1.clean(inplace=True)
# surf_obj2.clean(inplace=True)

# # Combine surfaces
# surface = surf_obj1 + surf_obj2

# # Save triangular surface mesh
# surface.save(base + out_folder + "option2_mesh_surface.vtk")

### Option 2 a) - Smooth sirface

In [5]:
# grid = pv.ImageData()
# grid.dimensions = numpy.array(mask.shape) + 1
# grid.spacing = (1.0, 1.0, 1.0)
# grid.origin = (0.0, 0.0, 0.0)

# grid.cell_data["values"] = mask.flatten(order="F")

# # Step 1: Extract surfaces for each domain
# surf_obj1 = grid.threshold([3-0.1, 3+0.1], scalars="values").extract_surface()
# surf_obj2 = grid.threshold([1-0.1, 1+0.1], scalars="values").extract_surface()

# # Smooth each domain individually to preserve boundaries
# surf_obj1_smooth = surf_obj1.smooth(n_iter=50, relaxation_factor=0.1)
# surf_obj2_smooth = surf_obj2.smooth(n_iter=50, relaxation_factor=0.1)

# # Combine smoothed surfaces
# surface_smooth = surf_obj1_smooth + surf_obj2_smooth
# surface_smooth_tri = surface_smooth.triangulate()


# # Optional: clean duplicates after smoothing
# surface_smooth_tri.clean(inplace=True)

# # Save
# surface_smooth_tri.save(base + out_folder + "option2_mesh_surface_smooth.vtk")

### Option 3

In [ ]:
# Assume 'mask' is your voxel array: 0=air, 1=obj1, 2=obj2
grid = pv.ImageData()
grid.dimensions = numpy.array(mask.shape) + 1
grid.spacing = (1.0, 1.0, 1.0)
grid.origin = (0.0, 0.0, 0.0)
grid.cell_data["values"] = mask.flatten(order="F")

# Step 1: Extract surfaces for each domain
surf_obj1 = grid.threshold([eye_id-0.1, eye_id+0.1], scalars="values").extract_surface()
surf_obj2 = grid.threshold([others_id[0]-0.1, others_id[0]+0.1], scalars="values").extract_surface()

# Step 2: Merge surfaces while snapping vertices at the interface
combined_surface = surf_obj1 + surf_obj2
combined_surface = combined_surface.triangulate()

combined_surface.clean(tolerance=1e-3, inplace=True)  # merge close vertices

# Step 3: Optional smoothing for a nicer surface
# combined_surface_smooth = combined_surface.smooth(n_iter=50, relaxation_factor=0.1)


combined_surface_smooth = combined_surface.smooth(
    n_iter=100,
    relaxation_factor=0.1,
    feature_smoothing=False,   # important!
    boundary_smoothing=True
)


# Step 4: Assign domain IDs per triangle using centroid sampling
centroids = combined_surface_smooth.cell_centers().points
domain_ids = []
for c in centroids:
    i,j,k = numpy.floor(c).astype(int)
    i = min(max(i,0), mask.shape[0]-1)
    j = min(max(j,0), mask.shape[1]-1)
    k = min(max(k,0), mask.shape[2]-1)
    domain_ids.append(mask[i,j,k])

combined_surface_smooth.cell_data["domain"] = numpy.array(domain_ids)



# Save
combined_surface_smooth.save(base + out_folder + "option3_mesh_surface_smooth.vtk")



/var/folders/l0/d5k4byxx3690l9b12qr7mm440000gp/T/ipykernel_7709/747365129.py:9: PyVistaFutureWarning: The default value of `algorithm` for the filter
`UnstructuredGrid.extract_surface` will change in the future. It currently defaults to
`'dataset_surface'`, but will change to `None`. Explicitly set the `algorithm` keyword to
silence this warning.
  surf_obj1 = grid.threshold([3-0.1, 3+0.1], scalars="values").extract_surface()
/var/folders/l0/d5k4byxx3690l9b12qr7mm440000gp/T/ipykernel_7709/747365129.py:10: PyVistaFutureWarning: The default value of `algorithm` for the filter
`UnstructuredGrid.extract_surface` will change in the future. It currently defaults to
`'dataset_surface'`, but will change to `None`. Explicitly set the `algorithm` keyword to
silence this warning.
  surf_obj2 = grid.threshold([1-0.1, 1+0.1], scalars="values").extract_surface()
